# 20 — MS-ASL Landmark Extraction

## Objetivo

Procesar todos los clips `.mp4` descargados de MS-ASL y extraer landmarks de manos
usando MediaPipe HandLandmarker. Pipeline idéntico al **Notebook 15** (WLASL2000).

## Estructura de entrada/salida

```
INPUT:  06_datasets/MS-ASL/videos/{split}/{label:04d}/{clip_id}.mp4
OUTPUT: 06_datasets/MS-ASL/landmarks/{split}/{gloss}/{clip_id}.npy
```

- `label` (int) → `gloss` (texto) via `MSASL_classes.json`
- Cada `.npy`: shape `(T, 21, 3)` con T <= 64 frames
- Normalización: centrado en muñeca (landmark 0), escala = dist(muñeca, dedo corazón base)
- Frames sin detección: interpolación lineal

## Características

- **Resume automático**: `_progress.json` + `_errors.json`, checkpoint cada 500 vídeos
- **Manifiesto construido desde disco**: escanea `videos/` sin necesidad de los JSON de anotación
- **metadata.json final**: conteo por clase y por split, mapeo label↔gloss

## Estimaciones

- ~16,816 vídeos disponibles en disco
- ~6.1 horas a 1.3s/vídeo (puede variar)
- ~500 MB de landmarks (estimado)

## Pipeline

```
MP4 video → OpenCV (sample 64 frames) → MediaPipe (21 landmarks x 3 coords)
          → Interpolate missing → Normalize (wrist-centered) → Save .npy
```

In [1]:
# === 1. Imports y Configuración ===

import mediapipe as mp
from mediapipe.tasks import python
from mediapipe.tasks.python import vision
import urllib.request
import os
import cv2
import numpy as np
import json
import time
from pathlib import Path
from tqdm import tqdm
from typing import List, Tuple, Dict, Optional, Set
from datetime import datetime
from collections import Counter, defaultdict
import warnings
warnings.filterwarnings('ignore')

# === Configuración ===
MAX_FRAMES = 64
MIN_DETECTION_RATE = 0.3   # Rechazar vídeos con < 30% detección de mano
CHECKPOINT_EVERY = 500     # Guardar progreso cada N vídeos

# === Rutas ===
BASE          = Path('/home/fransando/Dev/tfg-sign-language-recognition')
MSASL_DIR     = BASE / '06_datasets' / 'MS-ASL'
VIDEOS_DIR    = MSASL_DIR / 'videos'
CLASSES_PATH  = MSASL_DIR / 'MSASL_classes.json'
OUTPUT_ROOT   = MSASL_DIR / 'landmarks'
PROGRESS_FILE = OUTPUT_ROOT / '_progress.json'
ERROR_LOG_FILE= OUTPUT_ROOT / '_errors.json'
MODEL_PATH    = BASE / 'hand_landmarker.task'

OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

print(f'Fecha: {datetime.now().strftime("%Y-%m-%d %H:%M")}')
print(f'Videos dir:  {VIDEOS_DIR}')
print(f'Output root: {OUTPUT_ROOT}')
print(f'Checkpoint every: {CHECKPOINT_EVERY} vídeos')
print(f'Max frames: {MAX_FRAMES}')
print(f'Min detection rate: {MIN_DETECTION_RATE}')

Fecha: 2026-03-06 10:21
Videos dir:  /home/fransando/Dev/tfg-sign-language-recognition/06_datasets/MS-ASL/videos
Output root: /home/fransando/Dev/tfg-sign-language-recognition/06_datasets/MS-ASL/landmarks
Checkpoint every: 500 vídeos
Max frames: 64
Min detection rate: 0.3


In [2]:
# === 2. Cargar clases y construir manifiesto desde disco ===

# Cargar MSASL_classes.json: lista de 1000 glosses, índice = label
with open(CLASSES_PATH) as f:
    classes = json.load(f)

print(f'Clases cargadas: {len(classes)}')
print(f'  Primeras 5: {classes[:5]}')
print(f'  Últimas 5:  {classes[-5:]}')

# Construir manifiesto escaneando el sistema de archivos
# Estructura: videos/{split}/{label:04d}/{clip_id}.mp4
manifest = []
skipped_bad_dir = 0

for split in ['train', 'val', 'test']:
    split_dir = VIDEOS_DIR / split
    if not split_dir.exists():
        print(f'  AVISO: No existe {split_dir}')
        continue

    for label_dir in sorted(split_dir.iterdir()):
        if not label_dir.is_dir():
            continue
        # El directorio se llama "0830" → label = 830
        try:
            label = int(label_dir.name)
        except ValueError:
            skipped_bad_dir += 1
            continue

        gloss = classes[label] if label < len(classes) else f'unknown_{label}'

        for mp4_path in sorted(label_dir.glob('*.mp4')):
            clip_id = mp4_path.stem  # hash de 12 chars
            manifest.append({
                'clip_id'   : clip_id,
                'label'     : label,
                'gloss'     : gloss,
                'split'     : split,
                'video_path': str(mp4_path),
            })

print(f'\nManifiesto construido desde disco:')
print(f'  Total vídeos disponibles: {len(manifest)}')
if skipped_bad_dir > 0:
    print(f'  Directorios ignorados (nombre inválido): {skipped_bad_dir}')

split_counts = Counter(m['split'] for m in manifest)
gloss_counts = Counter(m['gloss'] for m in manifest)
print(f'\nPor split:')
for split in ['train', 'val', 'test']:
    print(f'  {split:5s}: {split_counts.get(split, 0):,} vídeos')

print(f'\nGlosses únicos: {len(gloss_counts)}')
print(f'Vídeos por gloss  — media: {np.mean(list(gloss_counts.values())):.1f} '
      f'  min: {min(gloss_counts.values())}  max: {max(gloss_counts.values())}')

# Estimación de tiempo
est_seconds = len(manifest) * 1.3
print(f'\nTiempo estimado: {est_seconds / 3600:.1f} horas')

Clases cargadas: 1000
  Primeras 5: ['hello', 'nice', 'teacher', 'eat', 'no']
  Últimas 5:  ['student center', 'cashier', 'flat tire', 'scientist', 'challenge']

Manifiesto construido desde disco:
  Total vídeos disponibles: 16816

Por split:
  train: 11,884 vídeos
  val  : 2,510 vídeos
  test : 2,422 vídeos

Glosses únicos: 1000
Vídeos por gloss  — media: 16.8   min: 2  max: 53

Tiempo estimado: 6.1 horas


In [3]:
# === 3. Sistema de checkpoint y resume ===

def load_progress(progress_file: Path) -> Tuple[Set[str], Dict]:
    """Carga el checkpoint de progreso previo."""
    if not progress_file.exists():
        return set(), {
            'total': 0, 'success': 0, 'failed': 0,
            'errors': {},
            'detection_rates': [],
            'by_split': {'train': 0, 'val': 0, 'test': 0},
        }
    with open(progress_file) as f:
        data = json.load(f)
    return set(data['completed_ids']), data['stats']


def save_progress(progress_file: Path, completed_ids: Set[str], stats: Dict):
    """Guardado atómico: escribe a .tmp y renombra."""
    progress_file.parent.mkdir(parents=True, exist_ok=True)
    tmp = progress_file.with_suffix('.tmp')
    save_stats = stats.copy()
    rates = save_stats.get('detection_rates', [])
    if rates:
        save_stats['detection_rate_summary'] = {
            'count': len(rates),
            'mean' : float(np.mean(rates)),
            'min'  : float(np.min(rates)),
            'max'  : float(np.max(rates)),
        }
    save_stats.pop('detection_rates', None)
    with open(tmp, 'w') as f:
        json.dump({'completed_ids': list(completed_ids), 'stats': save_stats}, f)
    tmp.rename(progress_file)


def load_error_log(error_log_file: Path) -> List[Dict]:
    if not error_log_file.exists():
        return []
    with open(error_log_file) as f:
        return json.load(f)


def save_error_log(error_log_file: Path, errors: List[Dict]):
    error_log_file.parent.mkdir(parents=True, exist_ok=True)
    with open(error_log_file, 'w') as f:
        json.dump(errors, f, indent=2)


# Cargar progreso previo
completed_ids, global_stats = load_progress(PROGRESS_FILE)
error_log = load_error_log(ERROR_LOG_FILE)
if 'detection_rates' not in global_stats:
    global_stats['detection_rates'] = []

# Filtrar manifiesto a lo que falta
remaining = [m for m in manifest if m['clip_id'] not in completed_ids]

print('Estado de resume:')
print(f'  Ya procesados: {len(completed_ids):,}')
print(f'  Pendientes:    {len(remaining):,}')
if completed_ids:
    print(f'  Exitosos hasta ahora: {global_stats["success"]:,}')
    print(f'  Fallidos hasta ahora: {global_stats["failed"]:,}')
    est_rem = len(remaining) * 1.3
    print(f'  Tiempo restante estimado: {est_rem / 3600:.1f} horas')

Estado de resume:
  Ya procesados: 0
  Pendientes:    16,816


In [4]:
# === 4. Inicializar MediaPipe HandLandmarker ===

# Descargar modelo si no existe
if not MODEL_PATH.exists():
    print('Descargando hand_landmarker.task...')
    url = 'https://storage.googleapis.com/mediapipe-models/hand_landmarker/hand_landmarker/float16/1/hand_landmarker.task'
    urllib.request.urlretrieve(url, str(MODEL_PATH))
    print('Modelo descargado.')

base_options = python.BaseOptions(model_asset_path=str(MODEL_PATH))
options = vision.HandLandmarkerOptions(
    base_options=base_options,
    num_hands=2,
    min_hand_detection_confidence=0.3,
    min_hand_presence_confidence=0.3,
    min_tracking_confidence=0.3,
)
detector = vision.HandLandmarker.create_from_options(options)
print('MediaPipe HandLandmarker inicializado.')

MediaPipe HandLandmarker inicializado.


INFO: Created TensorFlow Lite XNNPACK delegate for CPU.
W0000 00:00:1772788900.009207 1596636 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1772788900.023254 1596636 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.


In [5]:
# === 5. Funciones de procesamiento ===
# Idénticas al Notebook 15 (WLASL). No modificar.

def load_video_frames(video_path: Path, max_frames: int = 64) -> List[np.ndarray]:
    """
    Carga frames de un vídeo MP4.
    Si hay más de max_frames, submuestrea uniformemente.
    """
    cap = cv2.VideoCapture(str(video_path))
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

    if total_frames == 0:
        cap.release()
        return []

    if total_frames > max_frames:
        indices = set(np.linspace(0, total_frames - 1, max_frames, dtype=int))
    else:
        indices = set(range(total_frames))

    frames = []
    frame_idx = 0
    while True:
        ret, frame = cap.read()
        if not ret:
            break
        if frame_idx in indices:
            frames.append(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
        frame_idx += 1

    cap.release()
    return frames


def extract_landmarks_from_frame(
    frame: np.ndarray, hands_detector
) -> Optional[np.ndarray]:
    """
    Extrae landmarks de una mano en un frame.
    Devuelve array (21, 3) o None si no se detecta mano.
    """
    try:
        mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=frame)
        result = hands_detector.detect(mp_image)
        if result.hand_landmarks:
            landmarks = result.hand_landmarks[0]
            return np.array([[lm.x, lm.y, lm.z] for lm in landmarks])
    except Exception:
        pass
    return None


def interpolate_missing_frames(
    landmarks_list: List[Optional[np.ndarray]],
) -> np.ndarray:
    """
    Interpola linealmente los frames donde no se detectó mano.
    En los bordes copia el frame válido más cercano.
    """
    n_frames = len(landmarks_list)
    result = np.zeros((n_frames, 21, 3))
    valid_indices = [i for i, lm in enumerate(landmarks_list) if lm is not None]

    if len(valid_indices) == 0:
        return result

    for i in valid_indices:
        result[i] = landmarks_list[i]

    for i in range(n_frames):
        if landmarks_list[i] is not None:
            continue

        prev_valid = next((j for j in range(i - 1, -1, -1)
                           if landmarks_list[j] is not None), None)
        next_valid = next((j for j in range(i + 1, n_frames)
                           if landmarks_list[j] is not None), None)

        if prev_valid is not None and next_valid is not None:
            alpha = (i - prev_valid) / (next_valid - prev_valid)
            result[i] = (1 - alpha) * landmarks_list[prev_valid] + alpha * landmarks_list[next_valid]
        elif prev_valid is not None:
            result[i] = landmarks_list[prev_valid]
        elif next_valid is not None:
            result[i] = landmarks_list[next_valid]

    return result


def normalize_landmarks(landmarks: np.ndarray) -> np.ndarray:
    """
    Normalización invariante a posición y escala:
    - Centra en muñeca (landmark 0)
    - Escala por distancia muñeca → base dedo corazón (landmark 12)
    """
    normalized = np.copy(landmarks)
    for t in range(landmarks.shape[0]):
        frame = landmarks[t]
        wrist = frame[0]
        frame_centered = frame - wrist
        scale = np.linalg.norm(frame[12] - frame[0])
        normalized[t] = frame_centered / scale if scale > 1e-6 else frame_centered
    return normalized


def process_single_video(
    video_path: Path,
    hands_detector,
    max_frames: int = 64,
    min_detection_rate: float = 0.3,
) -> Tuple[Optional[np.ndarray], Dict]:
    """
    Procesa un vídeo completo: carga → landmarks → interpola → normaliza.
    Devuelve (array (T,21,3), stats) o (None, stats) si falla.
    """
    stats = {
        'success': False,
        'error': None,
        'sampled_frames': 0,
        'detected_frames': 0,
        'detection_rate': 0.0,
    }

    frames = load_video_frames(video_path, max_frames)
    if len(frames) == 0:
        stats['error'] = 'no_frames_loaded'
        return None, stats

    stats['sampled_frames'] = len(frames)

    landmarks_list = []
    for frame in frames:
        lm = extract_landmarks_from_frame(frame, hands_detector)
        landmarks_list.append(lm)
        if lm is not None:
            stats['detected_frames'] += 1

    stats['detection_rate'] = stats['detected_frames'] / stats['sampled_frames']

    if stats['detection_rate'] < min_detection_rate:
        stats['error'] = f'low_detection_rate_{stats["detection_rate"]:.2f}'
        return None, stats

    landmarks_interp = interpolate_missing_frames(landmarks_list)
    landmarks_norm   = normalize_landmarks(landmarks_interp)

    stats['success'] = True
    return landmarks_norm, stats


print('Funciones de procesamiento definidas.')

Funciones de procesamiento definidas.


---
## Bucle principal de extracción

Este cell puede tardar varias horas. Puedes interrumpir con **Kernel > Interrupt** 
y reanudar re-ejecutando el notebook desde la celda 3 (resume).

---

In [ ]:
# === 6. Bucle principal de procesamiento ===

if len(remaining) == 0:
    print('Todos los vídeos ya procesados. Nada que hacer.')
else:
    print(f'Procesando {len(remaining):,} vídeos...')
    print(f'Checkpoint cada {CHECKPOINT_EVERY} vídeos')
    print(f'Tiempo estimado: {len(remaining) * 1.3 / 3600:.1f} horas\n')

    start_time = time.time()

    for i, entry in enumerate(tqdm(remaining, desc='Extrayendo landmarks')):
        clip_id    = entry['clip_id']
        gloss      = entry['gloss']
        split      = entry['split']
        video_path = Path(entry['video_path'])

        try:
            landmarks, stats = process_single_video(
                video_path, detector, MAX_FRAMES, MIN_DETECTION_RATE
            )
        except Exception as e:
            stats = {'success': False, 'error': f'exception: {str(e)}',
                     'detection_rate': 0.0}
            landmarks = None
            error_log.append({
                'clip_id': clip_id, 'gloss': gloss,
                'split': split, 'error': str(e)
            })

        global_stats['total'] += 1

        if stats.get('success'):
            # Guardar landmarks
            out_dir = OUTPUT_ROOT / split / gloss
            out_dir.mkdir(parents=True, exist_ok=True)
            np.save(out_dir / f'{clip_id}.npy', landmarks)

            global_stats['success'] += 1
            global_stats['by_split'][split] = global_stats['by_split'].get(split, 0) + 1
            global_stats['detection_rates'].append(stats['detection_rate'])
        else:
            global_stats['failed'] += 1
            err = stats.get('error', 'unknown')
            global_stats['errors'][err] = global_stats['errors'].get(err, 0) + 1
            if 'exception' in str(stats.get('error', '')):
                error_log.append({
                    'clip_id': clip_id, 'gloss': gloss,
                    'split': split, 'error': stats['error']
                })

        completed_ids.add(clip_id)

        # Checkpoint periódico
        if (i + 1) % CHECKPOINT_EVERY == 0:
            save_progress(PROGRESS_FILE, completed_ids, global_stats)
            save_error_log(ERROR_LOG_FILE, error_log)
            elapsed = time.time() - start_time
            rate = (i + 1) / elapsed
            remaining_count = len(remaining) - (i + 1)
            eta_s = remaining_count / rate if rate > 0 else 0
            tqdm.write(
                f'\n[Checkpoint {i+1}/{len(remaining)}] '
                f'{global_stats["success"]} ok, {global_stats["failed"]} fallidos | '
                f'ETA: {eta_s/60:.0f} min'
            )

    # Guardado final
    save_progress(PROGRESS_FILE, completed_ids, global_stats)
    save_error_log(ERROR_LOG_FILE, error_log)

    elapsed_total = time.time() - start_time
    print(f'\nProcesamiento completado en {elapsed_total/3600:.1f} horas')

Procesando 16,816 vídeos...
Checkpoint cada 500 vídeos
Tiempo estimado: 6.1 horas



Extrayendo landmarks:   3%|▎         | 500/16816 [15:12<8:06:20,  1.79s/it] 


[Checkpoint 500/16816] 487 ok, 13 fallidos | ETA: 496 min


Extrayendo landmarks:   4%|▍         | 756/16816 [24:37<11:45:43,  2.64s/it]

In [ ]:
# === 7. Informe de procesamiento ===

print('=' * 70)
print('INFORME DE PROCESAMIENTO')
print('=' * 70)

total = global_stats['total']
print(f'\nTotal procesados: {total:,}')
print(f'  Exitosos: {global_stats["success"]:,} '
      f'({100 * global_stats["success"] / max(total, 1):.1f}%)')
print(f'  Fallidos: {global_stats["failed"]:,} '
      f'({100 * global_stats["failed"] / max(total, 1):.1f}%)')

print(f'\nPor split:')
for split, count in sorted(global_stats['by_split'].items()):
    print(f'  {split:5s}: {count:,}')

rates = global_stats.get('detection_rates', [])
if rates:
    rates_arr = np.array(rates)
    print(f'\nTasa de detección (vídeos exitosos):')
    print(f'  Media:   {rates_arr.mean():.1%}')
    print(f'  Mínima:  {rates_arr.min():.1%}')
    print(f'  Máxima:  {rates_arr.max():.1%}')
    print(f'  Mediana: {np.median(rates_arr):.1%}')

if global_stats['errors']:
    print(f'\nDesglose de errores:')
    for err, count in sorted(global_stats['errors'].items(), key=lambda x: -x[1])[:10]:
        print(f'  {err}: {count}')
    if len(global_stats['errors']) > 10:
        print(f'  ... y {len(global_stats["errors"]) - 10} tipos de error más')

In [ ]:
# === 8. Generar metadata.json ===

# Contar .npy por gloss por split
per_gloss_counts: Dict = {}
total_npy = 0

for split in ['train', 'val', 'test']:
    split_dir = OUTPUT_ROOT / split
    if not split_dir.exists():
        continue
    for gloss_dir in sorted(split_dir.iterdir()):
        if not gloss_dir.is_dir():
            continue
        gloss = gloss_dir.name
        count = len(list(gloss_dir.glob('*.npy')))
        total_npy += count
        if gloss not in per_gloss_counts:
            per_gloss_counts[gloss] = {'train': 0, 'val': 0, 'test': 0, 'total': 0}
        per_gloss_counts[gloss][split] = count
        per_gloss_counts[gloss]['total'] += count

# Construir mapeos de índice
glosses_sorted = sorted(per_gloss_counts.keys())
gloss_to_idx   = {g: i for i, g in enumerate(glosses_sorted)}
idx_to_gloss   = {str(i): g for i, g in enumerate(glosses_sorted)}

# Mapeo original: índice MSASL → gloss
msasl_label_to_gloss = {i: g for i, g in enumerate(classes)}
gloss_to_msasl_label = {g: i for i, g in enumerate(classes)}

metadata = {
    'name'       : 'msasl_landmarks',
    'source'     : 'MS-ASL',
    'description': 'MediaPipe Hand Landmarks extraídos de los clips del dataset MS-ASL',
    'landmark_shape'  : [21, 3],
    'max_frames'      : MAX_FRAMES,
    'min_detection_rate': MIN_DETECTION_RATE,
    'normalization'   : 'wrist_centered_scale_normalized',
    'interpolation'   : 'linear',
    'statistics': {
        'total_processed' : global_stats['total'],
        'total_success'   : global_stats['success'],
        'total_failed'    : global_stats['failed'],
        'success_rate'    : global_stats['success'] / max(global_stats['total'], 1),
        'mean_detection_rate': float(np.mean(global_stats['detection_rates']))
            if global_stats['detection_rates'] else 0.0,
        'by_split'        : global_stats['by_split'],
        'total_npy_files' : total_npy,
    },
    'num_classes'         : len(glosses_sorted),
    'glosses'             : glosses_sorted,
    'gloss_to_idx'        : gloss_to_idx,
    'idx_to_gloss'        : idx_to_gloss,
    'msasl_label_to_gloss': {str(k): v for k, v in msasl_label_to_gloss.items()},
    'gloss_to_msasl_label': gloss_to_msasl_label,
    'per_gloss_counts'    : per_gloss_counts,
    'errors'              : global_stats['errors'],
}

meta_path = OUTPUT_ROOT / 'metadata.json'
with open(meta_path, 'w') as f:
    json.dump(metadata, f, indent=2, ensure_ascii=False)

print(f'Metadata guardado en: {meta_path}')
print(f'\nResumen del dataset:')
print(f'  Clases:         {len(glosses_sorted)}')
print(f'  Total .npy:     {total_npy:,}')
print(f'  Train:          {sum(g["train"] for g in per_gloss_counts.values()):,}')
print(f'  Val:            {sum(g["val"]   for g in per_gloss_counts.values()):,}')
print(f'  Test:           {sum(g["test"]  for g in per_gloss_counts.values()):,}')

# Distribución de clips por clase
clips_per_class = [v['total'] for v in per_gloss_counts.values()]
print(f'\nClips por clase:')
print(f'  Media:  {np.mean(clips_per_class):.1f}')
print(f'  Mediana:{np.median(clips_per_class):.0f}')
print(f'  Mín:    {min(clips_per_class)}')
print(f'  Máx:    {max(clips_per_class)}')
print(f'  Clases con ≥ 20 clips: '
      f'{sum(1 for c in clips_per_class if c >= 20)} / {len(clips_per_class)}')

In [ ]:
# === 9. Verificación ===

import subprocess
import random

print('=' * 60)
print('VERIFICACIÓN')
print('=' * 60)

# Contar ficheros por split
for split in ['train', 'val', 'test']:
    split_dir = OUTPUT_ROOT / split
    if not split_dir.exists():
        print(f'{split}: directorio no encontrado')
        continue
    npy_files = list(split_dir.glob('**/*.npy'))
    gloss_dirs = [d for d in split_dir.iterdir() if d.is_dir()]
    print(f'{split:5s}: {len(npy_files):,} ficheros en {len(gloss_dirs):,} glosses')

# Verificar muestras aleatorias
all_npy = list(OUTPUT_ROOT.glob('**/[!_]*.npy'))
if all_npy:
    samples = random.sample(all_npy, min(5, len(all_npy)))
    print(f'\nVerificación de muestras aleatorias:')
    for sample_path in samples:
        data = np.load(sample_path)
        rel  = sample_path.relative_to(OUTPUT_ROOT)
        print(f'  {rel}: shape={data.shape}, dtype={data.dtype}, '
              f'rango=[{data.min():.3f}, {data.max():.3f}]')

# Uso de disco
result = subprocess.run(
    ['du', '-sh', str(OUTPUT_ROOT)],
    capture_output=True, text=True
)
print(f'\nUso total de disco: {result.stdout.strip().split()[0]}')

In [ ]:
# === 10. Limpieza ===

detector.close()
print('Recursos de MediaPipe liberados.')

print(f'\nFicheros de progreso (se pueden borrar tras verificar):')
print(f'  {PROGRESS_FILE}')
print(f'  {ERROR_LOG_FILE}')

---
## Resumen

### Estructura de salida
```
06_datasets/MS-ASL/landmarks/
├── metadata.json             (estadísticas, mapeos label↔gloss)
├── _progress.json            (checkpoint resume — borrar si se quiere)
├── _errors.json              (log de errores)
├── train/
│   ├── hello/
│   │   ├── a1b2c3d4e5f6.npy
│   │   └── ...
│   └── ...
├── val/
│   └── ...
└── test/
    └── ...
```

### Formato de cada .npy
- Shape: `(T, 21, 3)` donde T ≤ 64
- 21 landmarks MediaPipe, 3 coordenadas (x, y, z)
- Normalizado: centrado en muñeca, escala = dist(muñeca, base dedo corazón)
- Frames sin detección interpolados linealmente

### Directorio nombrado por gloss (texto), no por label numérico
- Ejemplo: `train/0830/` → `train/match/`
- El mapeo completo `label ↔ gloss` está en `metadata.json`

### Siguientes pasos
- **Notebook 21**: Crear subsets JSON de MS-ASL (100/200/500/1000 clases)
- **Notebook 22**: Entrenar GRU+Attention sobre MS-ASL
- **Notebook 23**: Combinar WLASL + MS-ASL en glosses compartidos